# Update Excel file into one platemap CSV per plate

The Excel file holds the layout for all five validation plates, with one treatment column per plate.
All plates share the same well layout (cells and controls), but each plate has its own three compounds.
We save one platemap per plate, named by the plate barcode so it can be matched to the profiles downstream.

In [ ]:
import pathlib

import pandas as pd


## Load in excel spreadsheet as pandas dataframe

In [ ]:
excel_df = pd.read_excel(
    pathlib.Path(
        "/home/jenna/mnt/Way_McKinsey_Cardiac_Fibrosis/Compound_Screen_data/hit_validation_2_cell_painting/Hit_Validation_2_Platemap_CORRECTED.xlsx"
    ),
    sheet_name="Platemap grid",
)

print(excel_df.shape)
excel_df.head()


(60, 11)


,WellRow,WellCol,well_position,heart_number,cell_type,heart_failure_type,treatment_plate 1,treatment_plate 2,treatment_plate 3,treatment_plate 4,treatment_plate 5
0,B,2,B02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
1,C,2,C02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
2,D,2,D02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
3,E,2,E02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
4,F,2,F02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO


## Update well row and column names to match the format of other columns

In [3]:
# Rename WellCol and WellRow columns to match other columns
excel_df = excel_df.rename(columns={'WellCol': 'well_column', 'WellRow': 'well_row'})

# Check example to see if update occurred properly
print(excel_df.shape)
excel_df.head()


(60, 11)


,well_row,well_column,well_position,heart_number,cell_type,heart_failure_type,treatment_plate 1,treatment_plate 2,treatment_plate 3,treatment_plate 4,treatment_plate 5
0,B,2,B02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
1,C,2,C02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
2,D,2,D02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
3,E,2,E02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
4,F,2,F02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO


## Fill NaN in heart failure type to "nonfailing"

In [4]:
# Fill NaN in heart_failure_type to "nonfailing"
excel_df['heart_failure_type'] = excel_df['heart_failure_type'].fillna('nonfailing')

# Check if columns have been properly updated
print(excel_df.shape)
excel_df.head()


(60, 11)


,well_row,well_column,well_position,heart_number,cell_type,heart_failure_type,treatment_plate 1,treatment_plate 2,treatment_plate 3,treatment_plate 4,treatment_plate 5
0,B,2,B02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
1,C,2,C02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
2,D,2,D02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
3,E,2,E02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO
4,F,2,F02,25,failing,dilated_cardiomyopathy,DMSO,DMSO,DMSO,DMSO,DMSO


## Map each plate number to its plate barcode

The barcodes match the folder names in each `Plate N` folder of the raw data.

In [5]:
# Plate number (from the Excel treatment columns) to plate barcode
plate_barcodes = {
    1: "CARD-CelIns-CX7_261030160001",
    2: "CARD-CelIns-CX7_261031160001",
    3: "CARD-CelIns-CX7_261101160003",
    4: "CARD-CelIns-CX7_261102080001",
    5: "CARD-CelIns-CX7_261102150001",
}

# Columns shared by all plates (everything except the per-plate treatment columns)
treatment_cols = [col for col in excel_df.columns if col.startswith("treatment_plate")]
shared_cols = [col for col in excel_df.columns if col not in treatment_cols]

# Make sure there is one treatment column per plate barcode
assert len(treatment_cols) == len(plate_barcodes), treatment_cols
print(treatment_cols)


['treatment_plate 1', 'treatment_plate 2', 'treatment_plate 3', 'treatment_plate 4', 'treatment_plate 5']


## Save one platemap CSV per plate

In [6]:
# make platemap output dir
output_dir = pathlib.Path("./platemaps")
output_dir.mkdir(exist_ok=True)

for plate_number, plate_barcode in plate_barcodes.items():
    plate_df = excel_df[shared_cols].copy()
    plate_df["treatment"] = excel_df[f"treatment_plate {plate_number}"]
    plate_df.insert(0, "plate_barcode", plate_barcode)
    plate_df.insert(1, "plate_number", plate_number)

    platemap_csv_path = output_dir / f"{plate_barcode}_platemap.csv"
    plate_df.to_csv(platemap_csv_path, index=False)

    compounds = sorted(plate_df.loc[plate_df["treatment"] != "DMSO", "treatment"].unique())
    print(f"Plate {plate_number} ({plate_barcode}): {plate_df.shape}, compounds: {compounds}")

plate_df.head()


Plate 1 (CARD-CelIns-CX7_261030160001): (60, 9), compounds: ['K278', 'K310', 'K543']
Plate 2 (CARD-CelIns-CX7_261031160001): (60, 9), compounds: ['K020', 'K756', 'K845']
Plate 3 (CARD-CelIns-CX7_261101160003): (60, 9), compounds: ['K208', 'K463', 'K779']
Plate 4 (CARD-CelIns-CX7_261102080001): (60, 9), compounds: ['K221', 'K318', 'K645']
Plate 5 (CARD-CelIns-CX7_261102150001): (60, 9), compounds: ['K191', 'K504', 'K687']


,plate_barcode,plate_number,well_row,well_column,well_position,heart_number,cell_type,heart_failure_type,treatment
0,CARD-CelIns-CX7_261102150001,5,B,2,B02,25,failing,dilated_cardiomyopathy,DMSO
1,CARD-CelIns-CX7_261102150001,5,C,2,C02,25,failing,dilated_cardiomyopathy,DMSO
2,CARD-CelIns-CX7_261102150001,5,D,2,D02,25,failing,dilated_cardiomyopathy,DMSO
3,CARD-CelIns-CX7_261102150001,5,E,2,E02,25,failing,dilated_cardiomyopathy,DMSO
4,CARD-CelIns-CX7_261102150001,5,F,2,F02,25,failing,dilated_cardiomyopathy,DMSO
